# <span style="color:red">  Monthly Data - EBC creation, FMB Quantiles

#### Import all our packages and functions

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

PROJECT_ROOT = Path.cwd().resolve().parents[0]
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
import importlib
import src.preference_factors.build_datasets as bd
import src.FamaMacBeth.betas as fmb_betas
import src.FamaMacBeth.pipeline as fmb_pipe
import src.FamaMacBeth.quantile as fmb_quantile
import src.FamaMacBeth.cross_sectional_regression as fmb_csr
import src.FamaMacBeth.pricing as fmb_pricing
import src.FamaMacBeth.backtest as fmb_backtest
import src.FamaMacBeth.plots as fmb_plots

# Reload the modules themselves
importlib.reload(bd)
importlib.reload(fmb_betas)
importlib.reload(fmb_pipe)
importlib.reload(fmb_quantile)
importlib.reload(fmb_csr)
importlib.reload(fmb_pricing)
importlib.reload(fmb_backtest)
importlib.reload(fmb_plots)


# <span style="color:red"> Create CSV's for Monthly EBC Weights, Beta Contributions, EBC returns, Preference Returns, Full preference and factors returns (Excess returns) </span>

In [ ]:
merged_df, full_merged_df = bd.build_all_dataset(
    returns_file="sp500_returns_monthly_with_tickers.csv",
    market_caps_file="sp500_market_caps_monthly.csv",
    ff_factors_file="ff_factors_monthly.csv",
    frequency="monthly"
)

ebc = merged_df["EBC"]
cw_ebc = merged_df["CW-EBC"]


In [ ]:
full_merged_df

In [ ]:
np.log1p(merged_df[['CW', 'EW', 'EBC']]).cumsum().plot(figsize=(10,6))
plt.title("Cumulative Log Excess Returns")
plt.show()

# <span style="color:red"> Get rolling betas for the two focus portfolios EBC and CW - EBC </span>

#### create factor returns

In [7]:
ebc = ebc.to_frame(name="EBC")
cw_ebc = cw_ebc.to_frame(name="CW-EBC")
factor_returns = pd.concat([ebc, cw_ebc], axis=1).dropna()
factor_returns.columns = ["EBC", "CW-EBC"]

#### get asset returns and make them excess

In [8]:
asset_returns = bd.build_returns_dataset("sp500_returns_monthly_with_tickers.csv", frequency="monthly")
ff_factors = bd.build_ff_dataset("ff_factors_monthly.csv", frequency="monthly")

asset_returns, ff_factors = asset_returns.align(ff_factors, join="inner", axis=0)
# Create the Excess Returns for the Assets
asset_excess_returns = asset_returns.sub(ff_factors["RF"], axis=0)

#### compute rolling betas for EBc and CW-EBC

In [ ]:
asset_excess_returns, factor_returns = asset_excess_returns.align(factor_returns, join="inner", axis=0)

betas_ebc = fmb_betas.compute_rolling_betas(asset_excess_returns, ebc, rolling_window=36, min_obs=10)
betas_cw_ebc = fmb_betas.compute_rolling_betas(asset_excess_returns, cw_ebc, rolling_window=36, min_obs=10)

# <span style="color:red"> Run full factor pipeline --> Two factor regressions FMB </span>

In [31]:
start_date = "1999-10-01"

## Filter all returns to match start date

In [32]:
asset_excess_returns_trimmed = asset_excess_returns.loc[start_date:]
factor_returns_trimmed = factor_returns.loc[start_date:]
betas_ebc_trimmed = betas_ebc.loc[start_date:]
betas_cw_ebc_trimmed = betas_cw_ebc.loc[start_date:]

In [ ]:
results = fmb_pipe.run_full_factor_pipeline(
    beta1_df=betas_ebc_trimmed,
    beta2_df=betas_cw_ebc_trimmed,
    sp500=asset_excess_returns_trimmed,
    EBC=factor_returns_trimmed["EBC"],
    Cap_EBC=factor_returns_trimmed["CW-EBC"],
    n_q1=3,
    n_q2=3,
    min_assets_per_cell=3,
    f1_name="EBC",
    f2_name="CW-EBC",
    frequency = 'monthly',
    formation_frequency='quarterly'
)

results["mean_grid"]

# <span style="color:red"> Backtest </span>

In [34]:
# ============================
# USER PARAMETERS
# ============================
#start_date = "1970-01-01"


# ============================
# REQUIRED INPUTS FROM PIPELINE
# ============================
members_df = results["members"]
n_q1, n_q2 = results["mean_grid"].shape

# ============================
# LOAD RETURNS / CAPS / RF
# ============================
DATA_RAW_DIR = Path.cwd().parent / "data" / "raw"
sp_500, sp_caps, rf = fmb_backtest.load_monthly_inputs(DATA_RAW_DIR) ### excess sp_500

In [ ]:
# ============================
# RUN BACKTEST
# ============================
quantile_portfolios_returns = fmb_backtest.backtest_quantile_portfolios(
    members_df=members_df,
    returns_df=sp_500,
    caps_df=sp_caps,
    n_q1=n_q1,
    n_q2=n_q2,
    weight="cap",
    frequency = 'monthly'
)
quantile_portfolios_returns = quantile_portfolios_returns.dropna(how='all')

In [36]:
rename_map = {
    "Q1_Q1": "EBC_High_Pref_High",
    "Q1_Q2": "EBC_High_Pref_Mid",
    "Q1_Q3": "EBC_High_Pref_Low",
    "Q2_Q1": "EBC_Mid_Pref_High",
    "Q2_Q2": "EBC_Mid_Pref_Mid",
    "Q2_Q3": "EBC_Mid_Pref_Low",
    "Q3_Q1": "EBC_Low_Pref_High",
    "Q3_Q2": "EBC_Low_Pref_Mid",
    "Q3_Q3": "EBC_Low_Pref_Low",
}

quantile_portfolios_returns = quantile_portfolios_returns.rename(columns=rename_map)


In [ ]:
quantile_portfolios_returns

# <span style="color:red"> PLOTS </span>

In [ ]:
full_merged_df.index = full_merged_df.index.to_timestamp(how="start")
full_merged_df.index.name = "date"


In [39]:
complete_df = full_merged_df.join(
    quantile_portfolios_returns,
    how="inner"
)


In [40]:
selected_columns = ['CW','EBC_High_Pref_High', 'EBC_High_Pref_Mid',
       'EBC_High_Pref_Low', 'EBC_Mid_Pref_High', 'EBC_Mid_Pref_Mid',
       'EBC_Mid_Pref_Low', 'EBC_Low_Pref_High', 'EBC_Low_Pref_Mid',
       'EBC_Low_Pref_Low']

In [ ]:
# ============================
# PLOTS
# ============================
fmb_plots.plot_cum_log_returns(
    complete_df[selected_columns],
    title=f"Monthly Cumulative Log Excess Returns ({n_q1}x{n_q2} Cap-Weighted)",
)

In [ ]:
# FM expected-return grid (annualized %)
implied_returns = fmb_plots.plot_expected_return_grid(results["pricing"], results["fm_table"], n_q1, n_q2)

In [ ]:
actual_returns = fmb_plots.plot_mean_grid(results["mean_grid"], annualize=True)

# <span style="color:red"> Save Files </span>

In [ ]:

DATA_PROCESSED_DIR_KEN = PROJECT_ROOT / "data" / "processed" / "Ken"

period = start_date[:4] + '_2024'
period

## Save implied monthly returns

In [45]:
implied_returns_renamed = implied_returns
implied_returns_renamed = implied_returns_renamed.rename(
    index={
        1: "High EBC",
        2: "Medium EBC",
        3: "Low EBC"
    },
    columns={
        1: "High Preference",
        2: "Medium Preference",
        3: "Low Preference"
    }
)
implied_returns_renamed.to_csv(DATA_PROCESSED_DIR_KEN/ f"monthly_{period}_implied_returns.csv")


## Save actual monthly returns

In [46]:
actual_returns_renamed = actual_returns

actual_returns_renamed = actual_returns_renamed.rename(
    index={
        1: "High EBC",
        2: "Medium EBC",
        3: "Low EBC"
    },
    columns={
        1: "High Preference",
        2: "Medium Preference",
        3: "Low Preference"
    }
)

actual_excel = actual_returns_renamed / 100

actual_excel.to_csv(DATA_PROCESSED_DIR_KEN/ f"monthly_{period}_actual_returns.csv")


## Save actual quantile time series

In [47]:
quantile_portfolios_returns_copy = quantile_portfolios_returns.copy()

quantile_portfolios_returns_copy.index = (
    quantile_portfolios_returns_copy.index.to_period("M")
)
quantile_portfolios_returns_copy.index.name = "date"

save_dir = DATA_PROCESSED_DIR / 'monthly'

quantile_portfolios_returns_copy.to_csv(save_dir / f"monthly_{period}_quantile_returns_3x3.csv")

# <span style="color:red"> Fitted GARCH </span>

In [26]:
from arch import arch_model


In [27]:
def fit_garch_model(returns):
    """
    Fit GARCH(1,1) with student t errors
    """

    returns = returns.dropna()

    # Scale to percentage
    returns_scaled = returns * 100

    am = arch_model(
        returns_scaled,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="t"
    )

    res = am.fit(disp="off")

    return res


In [ ]:
complete_df.columns

In [29]:
portfolio_name = 'CW'

In [ ]:
portfolio = complete_df[portfolio_name]

garch_res = fit_garch_model(portfolio)

print(garch_res.summary())


cond_vol = garch_res.conditional_volatility / 100  # scale back

plt.figure(figsize=(12,4))
plt.plot(cond_vol)
plt.title(f" {portfolio_name} - Conditional Volatility (GARCH)")
plt.ylabel("Volatility")
plt.grid(True)
plt.show()

mean_return = portfolio.mean()

cond_sharpe = mean_return / cond_vol

plt.figure(figsize=(12,4))
plt.plot(cond_sharpe)
plt.axhline(0, linestyle="--")
plt.title(f"{portfolio_name} Conditional Sharpe Ratio")
plt.grid(True)
plt.show()



